In [0]:
BRONZE_TABLE = "workspace.ev_streaming.bronze_charging_events"
GOLD_TABLE   = "workspace.ev_streaming.gold_station_windows"
GOLD_CHECKPOINT = "/Volumes/workspace/ev_streaming/raw_events/_checkpoint_gold"
print("Paths set ✅")

Paths set ✅


In [0]:
from pyspark.sql.functions import col, to_timestamp, window, count, sum, avg, round

bronze = (
    spark.read.table(BRONZE_TABLE)
        .withColumn("event_ts", to_timestamp(col("event_time")))
)

gold = (
    bronze
        .groupBy(
            window(col("event_ts"), "5 minutes"),
            col("station_id")
        )
        .agg(
            count("*").alias("sessions"),
            round(sum("energy_kwh"), 2).alias("total_energy_kwh"),
            round(sum("cost_inr"), 2).alias("total_revenue_inr"),
            round(avg("duration_min"), 1).alias("avg_duration_min")
        )
)

(gold.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(GOLD_TABLE))

print(f"Gold windowed aggregation complete ✅  ({gold.count()} rows)")

Gold windowed aggregation complete ✅  (10 rows)


In [0]:
gold = spark.read.table(GOLD_TABLE)
print(f"Gold rows: {gold.count()}")
gold.orderBy("window", "station_id").display()

Gold rows: 10


window,station_id,sessions,total_energy_kwh,total_revenue_inr,avg_duration_min
"List(2026-07-02T06:45:00.000Z, 2026-07-02T06:50:00.000Z)",STN-01,22,695.39,7057.98,57.9
"List(2026-07-02T06:45:00.000Z, 2026-07-02T06:50:00.000Z)",STN-02,16,459.77,4633.11,70.2
"List(2026-07-02T06:45:00.000Z, 2026-07-02T06:50:00.000Z)",STN-03,21,619.19,5955.51,61.5
"List(2026-07-02T06:45:00.000Z, 2026-07-02T06:50:00.000Z)",STN-04,18,433.5,4133.14,56.2
"List(2026-07-02T06:45:00.000Z, 2026-07-02T06:50:00.000Z)",STN-05,23,628.12,6267.81,59.6
"List(2026-07-02T06:50:00.000Z, 2026-07-02T06:55:00.000Z)",STN-01,4,134.7,1337.7,67.3
"List(2026-07-02T06:50:00.000Z, 2026-07-02T06:55:00.000Z)",STN-02,4,115.19,1162.97,48.3
"List(2026-07-02T06:50:00.000Z, 2026-07-02T06:55:00.000Z)",STN-03,8,324.21,3470.6,69.9
"List(2026-07-02T06:50:00.000Z, 2026-07-02T06:55:00.000Z)",STN-04,4,125.01,1298.07,51.3
"List(2026-07-02T06:50:00.000Z, 2026-07-02T06:55:00.000Z)",STN-05,5,124.44,1166.26,65.4


In [0]:
summary = spark.sql("""
    SELECT station_id,
           SUM(sessions)                    AS total_sessions,
           ROUND(SUM(total_energy_kwh), 2)  AS total_energy_kwh,
           ROUND(SUM(total_revenue_inr), 2) AS total_revenue_inr,
           ROUND(AVG(avg_duration_min), 1)  AS avg_duration_min
    FROM workspace.ev_streaming.gold_station_windows
    GROUP BY station_id
    ORDER BY total_energy_kwh DESC
""")
summary.display()

station_id,total_sessions,total_energy_kwh,total_revenue_inr,avg_duration_min
STN-03,29,943.4,9426.11,65.7
STN-01,26,830.09,8395.68,62.6
STN-05,28,752.56,7434.07,62.5
STN-02,20,574.96,5796.08,59.3
STN-04,22,558.51,5431.21,53.8
